In [30]:
!pip install --no-deps basic-pitch tensorflow-io-gcs-filesystem
!pip install mir_eval
!pip install --no-deps resampy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.9 MB/s eta 0:00:00a 0:00:01


In [16]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("AVISO: No hay GPU. Activa el acelerador en Settings → Accelerator → GPU.")

PyTorch version: 2.10.0+cu128
CUDA disponible: True
Dispositivo GPU: Tesla T4
VRAM total: 15.6 GB


In [17]:
import os
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("Datasets montados en /kaggle/input/:")
for d in sorted(INPUT_ROOT.iterdir()):
    print(f"  - {d.name}")


def find_wav_midi_pair(root: Path):
    """Devuelve el primer par (wav, midi) encontrado con el mismo nombre base."""
    for wav in root.rglob("*.wav"):
        for ext in (".midi", ".mid", ".MIDI", ".MID"):
            candidate = wav.with_suffix(ext)
            if candidate.exists():
                return wav, candidate
    return None, None


wav_path, midi_path = find_wav_midi_pair(INPUT_ROOT)
if wav_path is None:
    raise FileNotFoundError(
        "No se encontró ningún par .wav + .midi en /kaggle/input/. "
        "Asegúrate de haber añadido un dataset de MAESTRO al notebook (Add Input)."
    )

print()
print(f"Archivo seleccionado:")
print(f"  WAV : {wav_path}")
print(f"  MIDI: {midi_path}")
print(f"  Tamaño WAV: {wav_path.stat().st_size / 1e6:.1f} MB")

Datasets montados en /kaggle/input/:
  - datasets

Archivo seleccionado:
  WAV : /kaggle/input/datasets/alonhaviv/the-maestro-dataset-v3-0-0/maestro-v3.0.0/2017/MIDI-Unprocessed_059_PIANO059_MID--AUDIO-split_07-07-17_Piano-e_2-03_wav--1.wav
  MIDI: /kaggle/input/datasets/alonhaviv/the-maestro-dataset-v3-0-0/maestro-v3.0.0/2017/MIDI-Unprocessed_059_PIANO059_MID--AUDIO-split_07-07-17_Piano-e_2-03_wav--1.midi
  Tamaño WAV: 37.5 MB


In [18]:
import librosa
import soundfile as sf
import pretty_midi

DURATION_S = 30.0
TARGET_SR = 16000
WORK = Path("/kaggle/working")

# Audio
audio, sr = librosa.load(str(wav_path), sr=TARGET_SR, mono=True, duration=DURATION_S)
wav_out = WORK / "test_30s.wav"
sf.write(str(wav_out), audio, TARGET_SR)
print(f"Audio recortado: {len(audio) / TARGET_SR:.2f} s @ {TARGET_SR} Hz")

# MIDI ground truth
pm = pretty_midi.PrettyMIDI(str(midi_path))
gt_pm = pretty_midi.PrettyMIDI()
gt_inst = pretty_midi.Instrument(program=0, name="piano")
for inst in pm.instruments:
    for note in inst.notes:
        if note.start < DURATION_S:
            gt_inst.notes.append(
                pretty_midi.Note(
                    velocity=note.velocity,
                    pitch=note.pitch,
                    start=note.start,
                    end=min(note.end, DURATION_S),
                )
            )
gt_pm.instruments.append(gt_inst)
gt_out = WORK / "gt_30s.midi"
gt_pm.write(str(gt_out))
print(f"Ground truth MIDI: {len(gt_inst.notes)} notas en los primeros {DURATION_S} s")

Audio recortado: 30.00 s @ 16000 Hz
Ground truth MIDI: 398 notas en los primeros 30.0 s


## Inferencia con Kong  (piano_transcription_inference)

In [21]:
import time

from piano_transcription_inference import PianoTranscription, sample_rate as kong_sr

audio_kong, _ = librosa.load(str(wav_out), sr=kong_sr, mono=True)
print(f"Audio para Kong: {len(audio_kong) / kong_sr:.2f} s @ {kong_sr} Hz")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cargando modelo Kong et al. en {device}...")
transcriptor = PianoTranscription(device=device, checkpoint_path=None)

kong_out = WORK / "kong_out.midi"
print("Transcribiendo...")
t_start = time.time()
_ = transcriptor.transcribe(audio_kong, str(kong_out))
t_kong = time.time() - t_start
print(f"Tiempo de inferencia (Kong): {t_kong:.2f} s")

Audio para Kong: 30.00 s @ 16000 Hz
Cargando modelo Kong et al. en cuda...
Checkpoint path: /root/piano_transcription_inference_data/note_F1=0.9677_pedal_F1=0.9186.pth
Total size: ~165 MB


--2026-07-05 18:02:48--  https://zenodo.org/record/4034264/files/CRNN_note_F1%3D0.9677_pedal_F1%3D0.9186.pth?download=1
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.185.48.75, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/4034264/files/CRNN_note_F1=0.9677_pedal_F1=0.9186.pth [following]
--2026-07-05 18:02:48--  https://zenodo.org/records/4034264/files/CRNN_note_F1=0.9677_pedal_F1=0.9186.pth
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 171966578 (164M) [application/octet-stream]
Saving to: ‘/root/piano_transcription_inference_data/note_F1=0.9677_pedal_F1=0.9186.pth’

     0K .......... .......... .......... .......... ..........  0%  208K 13m28s
    50K .......... .......... .......... .......... ..........  0%  210K 13m24s
   100K .......... .......... .......... .......... ..........  0%  419

Using cuda for inference.
GPU number: 2
Transcribiendo...
Segment 0 / 5
Segment 1 / 5
Segment 2 / 5
Segment 3 / 5
Segment 4 / 5
Segment 5 / 5
Write out to /kaggle/working/kong_out.midi
Tiempo de inferencia (Kong): 2.99 s


## Inferencia con Basic Pitch (Spotify)

In [31]:
import shutil

from basic_pitch.inference import predict_and_save
from basic_pitch import ICASSP_2022_MODEL_PATH

bp_dir = WORK / "bp_out"
bp_dir.mkdir(exist_ok=True)

print("Transcribiendo con Basic Pitch...")
t_start = time.time()
predict_and_save(
    audio_path_list=[str(wav_out)],
    output_directory=str(bp_dir),
    save_midi=True,
    sonify_midi=False,
    save_model_outputs=False,
    save_notes=False,
    model_or_model_path=ICASSP_2022_MODEL_PATH,
)
t_bp = time.time() - t_start
print(f"Tiempo de inferencia (Basic Pitch): {t_bp:.2f} s")

bp_files = [f for f in os.listdir(bp_dir) if f.endswith((".mid", ".midi"))]
if not bp_files:
    raise RuntimeError("Basic Pitch no produjo un archivo MIDI de salida.")
bp_out = WORK / "bp_out.midi"
shutil.copy(bp_dir / bp_files[0], bp_out)
print(f"Archivo de salida: {bp_out}")

Transcribiendo con Basic Pitch...

Predicting MIDI for /kaggle/working/test_30s.wav...


I0000 00:00:1783274879.846284      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12682 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783274879.848376      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5




  Creating midi...
  💅 Saved to /kaggle/working/bp_out/test_30s_basic_pitch.mid
Tiempo de inferencia (Basic Pitch): 4.35 s
Archivo de salida: /kaggle/working/bp_out.midi


##  Evaluación con mir_eval

In [32]:
import numpy as np
import mir_eval


def extract_intervals_and_hz(midi_file, max_time=DURATION_S):
    pm = pretty_midi.PrettyMIDI(str(midi_file))
    intervals, pitches_hz = [], []
    for inst in pm.instruments:
        for note in inst.notes:
            if note.start < max_time:
                intervals.append([note.start, min(note.end, max_time)])
                pitches_hz.append(pretty_midi.note_number_to_hz(note.pitch))
    if not intervals:
        return np.zeros((0, 2)), np.zeros(0)
    return np.array(intervals), np.array(pitches_hz)


gt_intervals, gt_pitches = extract_intervals_and_hz(gt_out)
print(f"Ground truth: {len(gt_intervals)} notas")


def evaluate_model(name, midi_file, inference_time):
    est_i, est_p = extract_intervals_and_hz(midi_file)

    p_o, r_o, f1_o, _ = mir_eval.transcription.precision_recall_f1_overlap(
        gt_intervals, gt_pitches, est_i, est_p, offset_ratio=None
    )
    p_n, r_n, f1_n, _ = mir_eval.transcription.precision_recall_f1_overlap(
        gt_intervals, gt_pitches, est_i, est_p
    )

    return {
        "Modelo": name,
        "Notas detectadas": len(est_i),
        "F1_onset": round(f1_o * 100, 2),
        "P_onset": round(p_o * 100, 2),
        "R_onset": round(r_o * 100, 2),
        "F1_note (con offset)": round(f1_n * 100, 2),
        "Latencia (s)": round(inference_time, 2),
        "Latencia norm.": round(inference_time / DURATION_S, 3),
    }


results = [
    evaluate_model("Kong et al. (piano_transcription_inference)", kong_out, t_kong),
    evaluate_model("Basic Pitch (Spotify)", bp_out, t_bp),
]

for r in results:
    print(r)

Ground truth: 398 notas
{'Modelo': 'Kong et al. (piano_transcription_inference)', 'Notas detectadas': 398, 'F1_onset': 100.0, 'P_onset': 100.0, 'R_onset': 100.0, 'F1_note (con offset)': 97.24, 'Latencia (s)': 2.99, 'Latencia norm.': 0.1}
{'Modelo': 'Basic Pitch (Spotify)', 'Notas detectadas': 306, 'F1_onset': 81.25, 'P_onset': 93.46, 'R_onset': 71.86, 'F1_note (con offset)': 9.38, 'Latencia (s)': 4.35, 'Latencia norm.': 0.145}


In [33]:
import pandas as pd

col_order = [
    "Modelo",
    "F1_onset",
    "P_onset",
    "R_onset",
    "F1_note (con offset)",
    "Latencia (s)",
    "Latencia norm.",
]
df = pd.DataFrame(results)[col_order]

print("=" * 80)
print("TABLA DE RESULTADOS (Markdown) - COPIAR A literature_review_amt.md sección 7.1")
print("=" * 80)
print()
print(df.to_markdown(index=False))
print()
print("=" * 80)
print(f"Fragmento evaluado: {DURATION_S} s")
print(f"Fuente:  {wav_path}")
print(f"GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Notas ground truth: {len(gt_intervals)}")
print("=" * 80)

TABLA DE RESULTADOS (Markdown) - COPIAR A literature_review_amt.md sección 7.1

| Modelo                                      |   F1_onset |   P_onset |   R_onset |   F1_note (con offset) |   Latencia (s) |   Latencia norm. |
|:--------------------------------------------|-----------:|----------:|----------:|-----------------------:|---------------:|-----------------:|
| Kong et al. (piano_transcription_inference) |     100    |    100    |    100    |                  97.24 |           2.99 |            0.1   |
| Basic Pitch (Spotify)                       |      81.25 |     93.46 |     71.86 |                   9.38 |           4.35 |            0.145 |

Fragmento evaluado: 30.0 s
Fuente:  /kaggle/input/datasets/alonhaviv/the-maestro-dataset-v3-0-0/maestro-v3.0.0/2017/MIDI-Unprocessed_059_PIANO059_MID--AUDIO-split_07-07-17_Piano-e_2-03_wav--1.wav
GPU:     Tesla T4
Notas ground truth: 398


In [34]:
import json
from datetime import datetime

output = {
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "fragment_duration_s": DURATION_S,
    "source_wav": str(wav_path),
    "source_midi": str(midi_path),
    "ground_truth_notes": len(gt_intervals),
    "results": results,
}

out_json = WORK / "benchmark_results.json"
with open(out_json, "w") as f:
    json.dump(output, f, indent=2)

print(f"Resultados guardados en {out_json}")
print("Descárgalo desde el panel Output de Kaggle.")

Resultados guardados en /kaggle/working/benchmark_results.json
Descárgalo desde el panel Output de Kaggle.


/tmp/ipykernel_58/4142256709.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",
